# Exploración de datos (EDA)

Se ejecuta una sola vez; no forma parte del pipeline automatizado.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/FreddyForero1988/Proyecto-Final-Ap_Deep_Learning.git"


# 2. Clonar y entrar a la carpeta del proyecto (si no estás ya dentro)
if not os.path.exists("src"):
    repo_dir = REPO_URL.rstrip("/").split("/")[-1]
    if repo_dir.endswith(".git"):
        repo_dir = repo_dir[:-4]
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(repo_dir)
    # Si subiste la carpeta completa a GitHub, el proyecto queda en una subcarpeta
    if not os.path.exists("src"):
        subs = [d for d in os.listdir(".") if os.path.exists(os.path.join(d, "src"))]
        if not subs:
            raise FileNotFoundError("No encuentro la carpeta 'src' en el repositorio. "
                                    "Revisa qué archivos subiste a GitHub.")
        os.chdir(subs[0])

print("Carpeta de trabajo:", os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.config import RAW_DIR, GENRE_MAP
from src.data import download_raw

download_raw()
ratings = pd.read_csv(RAW_DIR / "ratings.csv")
books = pd.read_csv(RAW_DIR / "books.csv")
book_tags = pd.read_csv(RAW_DIR / "book_tags.csv")
tags = pd.read_csv(RAW_DIR / "tags.csv")
for n, d in [("ratings", ratings), ("books", books), ("book_tags", book_tags), ("tags", tags)]:
    print(n, d.shape)

## Calificaciones

In [ ]:
ratings["rating"].value_counts().sort_index().plot(kind="bar", title="Distribución de calificaciones")
plt.show()
print("Lectores:", ratings.user_id.nunique(), "| libros:", ratings.book_id.nunique())
print("Densidad de la matriz:", round(len(ratings) / (ratings.user_id.nunique() * ratings.book_id.nunique()), 4))

## Interacciones positivas (calificación ≥ 4)

In [ ]:
pos = ratings[ratings["rating"] >= 4]
per_user = pos.groupby("user_id").size()
print("Positivas:", len(pos), "de", len(ratings))
print(per_user.describe(percentiles=[.1, .25, .5, .75, .9]).round(1))
for k in [3, 5, 10, 20]:
    keep = per_user[per_user >= k].index
    print(f"mínimo {k}: {len(keep)} lectores, {pos.user_id.isin(keep).sum()} interacciones")
per_user.plot(kind="hist", bins=50, title="Interacciones positivas por lector")
plt.show()

## Popularidad de los libros

In [ ]:
per_book = pos.groupby("book_id").size().sort_values(ascending=False)
print(per_book.describe().round(1))
per_book.reset_index(drop=True).plot(title="Cola larga: positivas por libro (ordenadas)")
plt.show()

## Etiquetas: ruido en los estantes de usuarios

In [ ]:
top = (book_tags.merge(tags, on="tag_id").groupby("tag_name")["count"].sum()
       .sort_values(ascending=False).head(30))
print(top.to_string())

## Cobertura de géneros tras el mapeo (`GENRE_MAP`)

In [ ]:
from src.config import Config
from src.data import _item_genres

for drop in [(), ("fiction",)]:
    g = _item_genres(books, book_tags, tags, Config(drop_genres=drop))
    n = g.apply(lambda x: len(x) if isinstance(x, list) else 0)
    print("excluye", drop, "| libros sin géneros:", int((n == 0).sum()),
          "| media de géneros:", round(n.mean(), 2), "| géneros distintos:", len({x for l in g.dropna() for x in l}))